# Step 4. Transfer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/younesStrittmatter/sweetpea-starter/blob/main/tutorial/notebooks/4_transfer.ipynb)

The tutorial step without an assistant. Run the cells from top to bottom; the tables below are from one run, yours will differ where the design leaves things open.

In [ ]:
%pip install -q sweetpea pandas

**So far.** Step 2 made the trial before a factor. Step 3 made blocks that are mostly congruent or mostly incongruent, generated one by one. This step uses both ideas one level up: the mini-block is the unit, and the whole session is one design.

**The question.** Does adapting to frequent conflict make it harder to change the task?

**Prediction if it is one control setting.** Where conflict is frequent the current task is shielded more, so a change of task costs more: the switch cost is larger in the mostly incongruent context.

**Why word reading.** The switch cost is taken on word-reading mini-blocks. Word reading has almost no congruency effect, so the proportion itself hardly touches the trials the cost is measured on.

**The analysis.** Switch cost = response time in word-reading mini-blocks that follow colour naming (switch) minus response time in word-reading mini-blocks that follow word reading (repeat). Compare it between the two contexts.

### As before, plus the task and the context

Both are regular factors. Neither changes from trial to trial: each is held for a whole mini-block.

In [ ]:
import pandas as pd
from sweetpea import (Factor, DerivedLevel, WithinTrial, Transition, Exclude, MinimumTrials,
                      CrossBlock, synthesize_trials, CMSGen)

colours = ["red", "green", "blue"]

# regular factors: set freely on every trial
colour = Factor("colour", colours)      # the ink
word = Factor("word", colours)          # what the word says
from sweetpea import Nest

context = Factor("context", ["mostly congruent", "mostly incongruent"])  # new: held for a whole mini-block
task = Factor("task", ["name the colour", "read the word"])              # new: held for a whole mini-block

### New: does the task change from one mini-block to the next?

A transition factor, as in step 2. Here it will look back one mini-block, not one trial.

In [ ]:
def task_repeats(task):
    return task[0] == task[-1]


def task_switches(task):
    return task[0] != task[-1]


task_transition = Factor("task_transition", [
    DerivedLevel("repeat", Transition(task_repeats, [task])),
    DerivedLevel("switch", Transition(task_switches, [task])),
])

### New: what the context makes frequent

In a mostly congruent mini-block the congruent trials are the frequent ones, in a mostly incongruent mini-block the incongruent ones. One derived factor says that for both: it follows from the colour, the word and the context. The third argument of a level is its weight: 3 to 1 is three quarters.

In [ ]:
def frequent_here(colour, word, context):
    congruent = colour == word
    return congruent == (context == "mostly congruent")


def rare_here(colour, word, context):
    return not frequent_here(colour, word, context)


frequency = Factor("frequency", [
    DerivedLevel("frequent", WithinTrial(frequent_here, [colour, word, context]), 3),
    DerivedLevel("rare", WithinTrial(rare_here, [colour, word, context]), 1),
])

### New: a block inside a block

Two blocks, each with its own crossing:

- **within a mini-block**: the frequency crossed with the ink colour, 12 trials;
- **between mini-blocks**: the context, the task and the task transition, 8 mini-blocks, plus a first one that has no mini-block before it.

`Nest` puts the first inside the second: every combination of the outer block is held for one whole mini-block.

In [ ]:
mini_block = CrossBlock(design=[colour, word, frequency, context],
                        crossing=[frequency, colour],
                        constraints=[])
between_mini_blocks = CrossBlock(design=[context, task, task_transition],
                                 crossing=[context, task, task_transition],
                                 constraints=[])
block = Nest(outer_block=between_mini_blocks, inner_block=mini_block, alignment="post preamble")
session = pd.DataFrame(synthesize_trials(block, 1, CMSGen)[0])

# columns for counting
session["congruency"] = ["congruent" if colour == word else "incongruent"
                         for colour, word in zip(session["colour"], session["word"])]
session.insert(0, "block", session.index // 12 + 1)
overview = session.groupby("block")[["context", "task", "task_transition"]].first()
overview["congruent trials"] = session.groupby("block")["congruency"].apply(lambda c: (c == "congruent").sum())
display(overview)

,context,task,task_transition,congruent trials
block,,,,
1,mostly incongruent,name the colour,,6
2,mostly incongruent,name the colour,repeat,3
3,mostly congruent,name the colour,repeat,9
4,mostly incongruent,read the word,switch,3
5,mostly congruent,read the word,repeat,9
6,mostly incongruent,name the colour,switch,3
7,mostly congruent,read the word,switch,9
8,mostly incongruent,read the word,repeat,3
9,mostly congruent,name the colour,switch,9


One row per mini-block. The first has no task transition; it is the start. After it, every combination of context, task and transition comes once, and every mini-block has the proportion of its context: 9 or 3 of 12.

### Between mini-blocks: is everything crossed?

In [ ]:
counted = session[session["task_transition"] != ""]          # without the first mini-block
mini_blocks = counted.groupby("block")[["context", "task", "task_transition"]].first()
display(pd.crosstab([mini_blocks["context"], mini_blocks["task"]], mini_blocks["task_transition"]))

task_transition                     repeat  switch
context            task                           
mostly congruent   name the colour       1       1
                   read the word         1       1
mostly incongruent name the colour       1       1
                   read the word         1       1

### Within a mini-block: is the ink colour even?

In [ ]:
display(pd.crosstab([counted["block"], counted["context"]], [counted["congruency"], counted["colour"]]))

congruency               congruent           incongruent          
colour                        blue green red        blue green red
block context                                                     
2     mostly incongruent         1     1   1           3     3   3
3     mostly congruent           3     3   3           1     1   1
4     mostly incongruent         1     1   1           3     3   3
5     mostly congruent           3     3   3           1     1   1
6     mostly incongruent         1     1   1           3     3   3
7     mostly congruent           3     3   3           1     1   1
8     mostly incongruent         1     1   1           3     3   3
9     mostly congruent           3     3   3           1     1   1

### What is not in the design?

The adaptation that is supposed to carry over was built up in the mini-block before. Its context is not in the crossing. Count it for the mini-blocks the switch cost is taken from.

In [ ]:
mini_blocks["context_before"] = session.groupby("block")["context"].first().shift()
reading = mini_blocks[mini_blocks["task"] == "read the word"]
display(reading[["context", "task_transition", "context_before"]])

,context,task_transition,context_before
block,,,
4,mostly incongruent,switch,mostly congruent
5,mostly congruent,repeat,mostly incongruent
7,mostly congruent,switch,mostly incongruent
8,mostly incongruent,repeat,mostly congruent


**Check.** Both crossings are exact. What is thin? Each cell of the comparison is one mini-block of 12 trials; more need a longer session or several sequences. Is the context of the mini-block before the same for switch and repeat? It is left to chance: run the block cell again and look. Should it be a factor too, and how many mini-blocks would that take?

**Try.** Add `MinimumTrials(17)` to the outer block: it counts mini-blocks, so that is two rounds plus the start.

**The idea.** Each question added a factor, and each factor multiplies the design. Nesting keeps both levels counterbalanced at once. What it leaves open is still a choice, and which choice is acceptable depends on what you compare.